# `train_recommend`

Module 2 - CROP RECOMMENDATION (22-class classification from N, P, K, temp, humidity, pH, rainfall).

The public dataset is very clean (~99% accuracy for almost any model), so plain accuracy is not a
meaningful result. Besides standard metrics this module reports:
  * noise robustness      - Gaussian noise (k x feature std) added to the TEST features
  * range extrapolation   - train without each class's most extreme 15% of one feature, test on them
  * a 'robust score' used to pick the deployed model (mean of CV acc, noise@0.25, range-holdout acc)

> Part of the crop yield & recommendation project. Imported by `run_all.py` (via the notebook loader in `src/__init__.py`) or runnable directly here.

In [ ]:
# Setup: make the project root importable when this notebook is opened directly in Jupyter
import sys
from pathlib import Path
_root = Path.cwd().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from src import config as C
from src.utils import have, savefig, set_seed

warnings.filterwarnings("ignore")
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
NOISE_LEVELS = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
RANGE_FEATURES = ["temperature", "humidity", "rainfall"]

In [ ]:
def make_models(fast=False):
    n = 60 if fast else 300
    m = {
        "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "SVM(RBF)": make_pipeline(StandardScaler(), SVC(C=10, probability=True, random_state=C.SEED)),
        "DecisionTree": DecisionTreeClassifier(max_depth=12, random_state=C.SEED),
        "RandomForest": RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=C.SEED),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=n, learning_rate=0.08,
                                                               random_state=C.SEED),
    }
    if have("xgboost"):
        from xgboost import XGBClassifier
        m["XGBoost"] = XGBClassifier(n_estimators=n, max_depth=6, learning_rate=0.08, subsample=0.8,
                                     colsample_bytree=0.8, n_jobs=-1, random_state=C.SEED, verbosity=0)
    if have("lightgbm"):
        from lightgbm import LGBMClassifier
        m["LightGBM"] = LGBMClassifier(n_estimators=n, learning_rate=0.08, num_leaves=31, n_jobs=-1,
                                       random_state=C.SEED, verbose=-1)
    if have("catboost"):
        from catboost import CatBoostClassifier
        m["CatBoost"] = CatBoostClassifier(iterations=n, depth=6, learning_rate=0.1, verbose=0,
                                           random_seed=C.SEED, thread_count=-1)
    ens = [(k, clone(m[k])) for k in ("RandomForest", "HistGradientBoosting", "SVM(RBF)") if k in m]
    m["Voting(soft)"] = VotingClassifier(estimators=ens, voting="soft")
    return m

In [ ]:
def noisy(X, std, k, rng):
    return X + rng.normal(0, 1, X.shape) * std.values * k

In [ ]:
def main(fast=False):
    print("[3/4] Crop recommendation")
    set_seed()
    df = pd.read_csv(C.RECO_CLEAN)
    le = LabelEncoder().fit(df["crop"])
    X, y = df[FEATURES], pd.Series(le.transform(df["crop"]), index=df.index)
    models = make_models(fast)

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=C.SEED)
    std = Xtr.std()
    skf = list(StratifiedKFold(C.CV_FOLDS, shuffle=True, random_state=C.SEED).split(X, y))
    rng = np.random.default_rng(C.SEED)

    # range-extrapolation splits: per class, hold out the top 15% of one feature
    range_splits = {}
    for f in RANGE_FEATURES:
        thr = df.groupby("crop")[f].transform(lambda s: s.quantile(0.85))
        range_splits[f] = (df[f] > thr).values

    rows, noise_rows, fold_acc, fitted, preds = [], [], {}, {}, {}
    for name, mdl in models.items():
        t0 = time.time()
        fit = clone(mdl).fit(Xtr, ytr)
        fitted[name] = fit
        p = fit.predict(Xte)
        preds[name] = p
        acc, f1 = accuracy_score(yte, p), f1_score(yte, p, average="macro")
        accs = [accuracy_score(y.iloc[te], clone(mdl).fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te]))
                for tr, te in skf]
        fold_acc[name] = accs

        for k in NOISE_LEVELS:
            accn = np.mean([accuracy_score(yte, fit.predict(noisy(Xte, std, k, rng))) for _ in range(5)]) if k else acc
            noise_rows.append({"Model": name, "noise_k": k, "accuracy": accn})

        rng_acc = {}
        for f, mask in range_splits.items():
            m2 = clone(mdl).fit(X[~mask], y[~mask])
            rng_acc[f"Range_{f}"] = accuracy_score(y[mask], m2.predict(X[mask]))
        n25 = [r["accuracy"] for r in noise_rows if r["Model"] == name and r["noise_k"] == 0.25][0]
        rows.append({"Model": name, "Accuracy": acc, "Macro_F1": f1,
                     "CV_Acc_mean": np.mean(accs), "CV_Acc_std": np.std(accs),
                     "Noise_0.25_Acc": n25, **rng_acc,
                     "Robust_score": np.mean([np.mean(accs), n25, np.mean(list(rng_acc.values()))]),
                     "Time_s": time.time() - t0})
        print(f"  {name:<22} acc={acc:.4f} F1={f1:.4f} CV={np.mean(accs):.4f} "
              f"noise@.25={n25:.3f} range-holdout={np.mean(list(rng_acc.values())):.3f}")

    res = pd.DataFrame(rows).sort_values("Robust_score", ascending=False).reset_index(drop=True)
    res.to_csv(C.REPORTS / "reco_results.csv", index=False)
    pd.DataFrame(noise_rows).to_csv(C.REPORTS / "reco_noise_robustness.csv", index=False)
    best = res.loc[0, "Model"]
    print(f"  BEST (robust score): {best}")

    # significance of CV accuracy: best vs others
    sig = []
    for name in res["Model"][1:]:
        a, b = np.array(fold_acc[best]), np.array(fold_acc[name])
        try:
            w = stats.wilcoxon(a, b).pvalue
        except ValueError:
            w = np.nan
        sig.append({"Best": best, "Compared_to": name, "mean_acc_diff": float((a - b).mean()),
                    "paired_t_p": stats.ttest_rel(a, b).pvalue, "wilcoxon_p": w})
    pd.DataFrame(sig).to_csv(C.REPORTS / "reco_significance.csv", index=False)

    # figures: confusion matrix, most-confused pairs, robustness curve
    cm = confusion_matrix(yte, preds[best])
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=le.classes_, yticklabels=le.classes_, cbar=False)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix - {best}")
    savefig("reco_confusion_matrix.png")

    off = cm.copy(); np.fill_diagonal(off, 0)
    pairs = sorted(((le.classes_[i], le.classes_[j], int(off[i, j]))
                    for i in range(len(off)) for j in range(len(off)) if off[i, j] > 0),
                   key=lambda t: -t[2])[:10]
    pd.DataFrame(pairs, columns=["actual", "predicted_as", "count"]).to_csv(C.REPORTS / "reco_confused_pairs.csv", index=False)

    nr = pd.DataFrame(noise_rows)
    plt.figure(figsize=(8, 4.8))
    sns.lineplot(data=nr, x="noise_k", y="accuracy", hue="Model", marker="o")
    plt.xlabel("Noise level (k x feature std)"); plt.ylabel("Test accuracy"); plt.title("Robustness to input noise")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    savefig("reco_noise_robustness.png")

    # refit best on all data, save bundle
    final = clone(models[best]).fit(X, y)
    joblib.dump({"model": final, "classes": le.classes_.tolist(), "features": FEATURES,
                 "ranges": {f: [float(X[f].min()), float(X[f].max()), float(X[f].median())] for f in FEATURES}},
                C.MODELS / "recommend_best.joblib")
    (C.MODELS / "recommend_meta.json").write_text(json.dumps({"best_model": best, **res.iloc[0].to_dict()}, indent=2, default=float))
    print("  saved models/recommend_best.joblib")
    return res

In [ ]:
if __name__ == "__main__":
    main()